In [2]:
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm sentencepiece

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
SAVE_DIR = "/content/drive/MyDrive/thesis_results/verification/llama-2-7b"
os.makedirs(SAVE_DIR, exist_ok=True)

DRIVE_SCALES = "/content/drive/MyDrive/thesis_results/act_scales/llama-2-7b.pt"
REPO_SCALES  = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/llama-2-7b.pt"
assert os.path.exists(DRIVE_SCALES), f"missing: {DRIVE_SCALES} — run generate_act_scales_cells.md first."
os.makedirs(os.path.dirname(REPO_SCALES), exist_ok=True)
shutil.copy2(DRIVE_SCALES, REPO_SCALES)

print("max scales :", REPO_SCALES)
!nvidia-smi

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 251 (delta 105), reused 211 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (251/251), 5.31 MiB | 23.42 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 14.97 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
Mounted at /content/drive
max scales : /content/llm-quantization-thesis/smoothquant_repo/act_scale

In [3]:
import re, torch, json

SCALES_PATH = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/llama-2-7b.pt"
SAVE_DIR    = "/content/drive/MyDrive/thesis_results/verification/llama-2-7b"
ALPHA_MIN, ALPHA_MAX = 0.75, 0.95   # focused range — anchored near paper's Llama-2 optimum (α=0.85)

raw = torch.load(SCALES_PATH, map_location="cpu")
LAYER_RE = re.compile(r"model\.layers\.(\d+)\.(.+)")

sev_by_layer = {}
for name, vec in raw.items():
    m = LAYER_RE.match(name)
    if not m:
        continue
    suffix = m.group(2)
    if suffix not in ("self_attn.q_proj", "mlp.gate_proj"):
        continue
    layer = int(m.group(1))
    v = vec.float().abs()
    s = (v.max() / v.median().clamp(min=1e-12)).item()
    sev_by_layer.setdefault(layer, []).append(s)

layers = sorted(sev_by_layer.keys())
sev = torch.tensor([sum(sev_by_layer[l]) / len(sev_by_layer[l]) for l in layers])

# Linear normalisation — original OPT-style sev/sev_max.
sev_norm = sev / sev.max()
alpha_per_layer = (ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * sev_norm).tolist()

# --- Fallback option -------------------------------------------------------
# If even [0.7, 0.95] underperforms, the original wider range was:
#     ALPHA_MIN, ALPHA_MAX = 0.5, 0.9
# Switching to [0.7, 0.95] keeps the ordering driven by calibration severity
# but anchors the range near the paper's Llama-2 optimum (α=0.85). The recipe
# is still parameter-free *per model* — the range is a once-set architectural
# default for the Llama family, not a per-model grid search.
# ---------------------------------------------------------------------------

assert len(alpha_per_layer) == 32, f"expected 32 layers, got {len(alpha_per_layer)}"
print(f"per-layer α range: [{min(alpha_per_layer):.3f}, {max(alpha_per_layer):.3f}]  (32 layers)")
print(f"severity spread (max/min): {sev.max().item() / sev.min().item():.2f}×")
print()
print("layer  severity   α(l)")
for l, s, a in zip(layers, sev.tolist(), alpha_per_layer):
    print(f"  {l:2d}    {s:7.2f}    {a:.3f}")

# Persist the schedule so it survives runtime disconnects.
with open(f"{SAVE_DIR}/llama-2-7b_alpha_schedule.json", "w") as f:
    json.dump({
        "layers": layers,
        "severity": sev.tolist(),
        "alpha_per_layer": alpha_per_layer,
        "alpha_range": [ALPHA_MIN, ALPHA_MAX],
        "normalisation": "linear",
    }, f, indent=2)
print(f"\nschedule saved -> {SAVE_DIR}/llama-2-7b_alpha_schedule.json")

per-layer α range: [0.768, 0.950]  (32 layers)
severity spread (max/min): 11.13×

layer  severity   α(l)
   0      65.80    0.950
   1      27.22    0.833
   2      11.99    0.786
   3       6.86    0.771
   4      10.06    0.781
   5       8.66    0.776
   6       8.35    0.775
   7       8.61    0.776
   8      11.95    0.786
   9      14.68    0.795
  10      14.88    0.795
  11      11.78    0.786
  12      12.55    0.788
  13      11.65    0.785
  14      10.98    0.783
  15      11.12    0.784
  16      10.35    0.781
  17       8.41    0.776
  18       8.46    0.776
  19       8.12    0.775
  20       7.92    0.774
  21       7.72    0.773
  22       7.57    0.773
  23       6.81    0.771
  24       7.57    0.773
  25       6.54    0.770
  26       7.32    0.772
  27       5.91    0.768
  28       6.01    0.768
  29       6.44    0.770
  30       6.79    0.771
  31       9.08    0.778

schedule saved -> /content/drive/MyDrive/thesis_results/verification/llama-2-7b/llama-2-7b_alp

In [4]:
%%writefile /content/smooth_per_layer_llama.py
"""Per-layer α extension of smoothquant.smooth.smooth_lm (Llama)."""
import re
import torch
from transformers.models.llama.modeling_llama import LlamaDecoderLayer
from smoothquant.smooth import smooth_ln_fcs_llama_like

LAYER_RE = re.compile(r"model\.layers\.(\d+)$")

@torch.no_grad()
def smooth_lm_per_layer_llama(model, scales, alpha_schedule):
    if not isinstance(alpha_schedule, (list, tuple)):
        raise TypeError("alpha_schedule must be a list/tuple of floats")
    for name, module in model.named_modules():
        if not isinstance(module, LlamaDecoderLayer):
            continue
        m = LAYER_RE.match(name)
        if m is None:
            continue
        layer_idx = int(m.group(1))
        alpha = float(alpha_schedule[layer_idx])

        attn_ln = module.input_layernorm
        qkv = [module.self_attn.q_proj, module.self_attn.k_proj, module.self_attn.v_proj]
        qkv_input_scales = scales[name + ".self_attn.q_proj"]
        smooth_ln_fcs_llama_like(attn_ln, qkv, qkv_input_scales, alpha)

        ffn_ln = module.post_attention_layernorm
        fcs = [module.mlp.gate_proj, module.mlp.up_proj]
        fcs_input_scales = scales[name + ".mlp.gate_proj"]
        smooth_ln_fcs_llama_like(ffn_ln, fcs, fcs_input_scales, alpha)

Writing /content/smooth_per_layer_llama.py


In [5]:
import sys
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")

import torch, torch.nn as nn, json, time, tqdm, os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from smoothquant.smooth import smooth_lm
from smoothquant.fake_quant import quantize_model
from smooth_per_layer_llama import smooth_lm_per_layer_llama

MODEL = "NousResearch/Llama-2-7b-hf"
SCALES_PATH = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/llama-2-7b.pt"
SAVE_DIR    = "/content/drive/MyDrive/thesis_results/verification/llama-2-7b"


class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=40):
        self.dataset = tokenizer("\n\n".join(dataset["text"]), return_tensors="pt").input_ids.to(device)
        self.n_samples = n_samples
    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        nlls = []
        n = self.n_samples
        for i in tqdm.tqdm(range(n), desc="PPL"):
            batch = self.dataset[:, (i * 2048):((i + 1) * 2048)].to(model.device)
            logits = model(batch).logits
            shift_logits = logits[:, :-1, :].contiguous().float()
            shift_labels = self.dataset[:, (i * 2048):((i + 1) * 2048)][:, 1:]
            loss = nn.CrossEntropyLoss()(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            nlls.append(loss.float() * 2048)
        return torch.exp(torch.stack(nlls).sum() / (n * 2048))


print("Loading tokenizer + dataset + scales...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL, use_fast=False)
dataset    = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
evaluator  = Evaluator(dataset, tokenizer, "cuda")
act_scales = torch.load(SCALES_PATH)

C_QPARAMS = dict(weight_quant="per_channel", act_quant="per_token", quantize_bmm_input=True)
PAPER_ALPHA = 0.85  # SmoothQuant paper Table 7 — Llama-2-7B row

RUNS = [
    {"label": "1_FP16",          "smooth": "none",                                "qparams": None},
    {"label": f"2_C_max_a{PAPER_ALPHA}", "smooth": ("max", PAPER_ALPHA),          "qparams": C_QPARAMS},
    {"label": "3_C_perlayer",    "smooth": ("perlayer", alpha_per_layer),         "qparams": C_QPARAMS},
]

results = []
for i, run in enumerate(RUNS, 1):
    print(f"\n{'='*60}\n  Run {i}/{len(RUNS)}: {run['label']}\n{'='*60}")
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

    sm = run["smooth"]
    if sm == "none":
        pass
    elif sm[0] == "max":
        smooth_lm(model, act_scales, sm[1])
    elif sm[0] == "perlayer":
        smooth_lm_per_layer_llama(model, act_scales, sm[1])
    else:
        raise ValueError(f"unknown smooth spec: {sm}")

    if run["qparams"] is not None:
        model = quantize_model(model, **run["qparams"])

    ppl = evaluator.evaluate(model).item()
    elapsed = time.time() - t0
    print(f">>> {run['label']}: PPL = {ppl:.4f}  ({elapsed:.0f}s)")

    rec = {
        "model": MODEL,
        "label": run["label"],
        "smooth": (sm if isinstance(sm, str) else (sm[0] if sm[0] != "perlayer" else "perlayer")),
        "qparams": run["qparams"],
        "ppl": round(ppl, 4),
        "seconds": round(elapsed, 1),
    }
    results.append(rec)
    with open(f"{SAVE_DIR}/llama-2-7b_{run['label']}.json", "w") as f:
        json.dump(rec, f, indent=2)

    del model
    torch.cuda.empty_cache()

with open(f"{SAVE_DIR}/llama-2-7b_summary.json", "w") as f:
    json.dump({"results": results, "alpha_per_layer": alpha_per_layer}, f, indent=2)
print(f"\nsaved -> {SAVE_DIR}/llama-2-7b_summary.json")

Loading tokenizer + dataset + scales...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

PPL: 100%|██████████| 40/40 [00:08<00:00,  4.56it/s]


>>> 2_C_max_a0.85: PPL = 5.8658  (268s)

  Run 3/3: 3_C_perlayer


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PPL: 100%|██████████| 40/40 [00:08<00:00,  4.57it/s]


>>> 3_C_perlayer: PPL = 5.8630  (268s)

saved -> /content/drive/MyDrive/thesis_results/verification/llama-2-7b/llama-2-7b_summary.json


In [6]:
def ppl(label):
    return next(r for r in results if r["label"] == label)["ppl"]

PAPER_FP16, PAPER_SQ, PAPER_ALPHA = 5.474, 5.515, 0.85
our_fp16 = ppl("1_FP16")
our_paper = ppl(f"2_C_max_a{PAPER_ALPHA}")
our_alpha_l = ppl("3_C_perlayer")

print(f"\n{'='*78}")
print(f"  Llama-2-7B — WikiText-2 PPL — paper-comparable table")
print(f"{'='*78}")
print(f"\n{'config':<30} {'ours':>10} {'paper':>10} {'Δ ours−paper':>14}")
print("-" * 70)
print(f"{'FP16':<30} {our_fp16:>10.4f} {PAPER_FP16:>10.4f} {our_fp16 - PAPER_FP16:>+14.4f}")
print(f"{'C + max α=0.85 (paper cfg)':<30} {our_paper:>10.4f} {PAPER_SQ:>10.4f} {our_paper - PAPER_SQ:>+14.4f}")
print(f"{'C + per-layer α (ours)':<30} {our_alpha_l:>10.4f} {'—':>10} {'—':>14}")

print(f"\nDeltas vs OUR FP16 (within-session, noise-free):")
print(f"  C max α=0.85       − FP16  = {our_paper   - our_fp16:+.4f}")
print(f"  C per-layer α      − FP16  = {our_alpha_l - our_fp16:+.4f}")
print(f"  C per-layer        − C max = {our_alpha_l - our_paper:+.4f}  (negative → α(l) wins)")

print(f"\nPaper-gap reference:")
print(f"  paper W8A8 − paper FP16    = {PAPER_SQ - PAPER_FP16:+.4f}  (Table 7)")

shift = our_fp16 - PAPER_FP16
print(f"\nProtocol-shift diagnostic:")
print(f"  our FP16 − paper FP16  = {shift:+.4f}")
if abs(shift) < 0.03:
    print(f"  → protocols match; can cite paper numbers directly.")
else:
    print(f"  → protocols differ by ~{shift:+.3f} PPL; compare *within-session* deltas only.")


  Llama-2-7B — WikiText-2 PPL — paper-comparable table

config                               ours      paper   Δ ours−paper
----------------------------------------------------------------------
FP16                               5.8242     5.4740        +0.3502
C + max α=0.85 (paper cfg)         5.8658     5.5150        +0.3508
C + per-layer α (ours)             5.8630          —              —

Deltas vs OUR FP16 (within-session, noise-free):
  C max α=0.85       − FP16  = +0.0416
  C per-layer α      − FP16  = +0.0388
  C per-layer        − C max = -0.0028  (negative → α(l) wins)

Paper-gap reference:
  paper W8A8 − paper FP16    = +0.0410  (Table 7)

Protocol-shift diagnostic:
  our FP16 − paper FP16  = +0.3502
  → protocols differ by ~+0.350 PPL; compare *within-session* deltas only.
